In [3]:
%pip install pandas             # Para a leitura e o tratamento dos dados
%pip install openpyxl           # Para leitura de arquivos do excel (.xlsx)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

## Extraindo os dados

In [5]:
# Definindo o padrão de colunas (tudo em minúsculo) para forçar em todos os arquivos (principalmente em 2017 que está sem header)
colunas_padrao = [
    'anocalendario', 'dataarquivamento', 'dataabertura', 'codigoregiao', 'regiao',
    'uf', 'strrazaosocial', 'strnomefantasia', 'tipo', 'numerocnpj', 'radicalcnpj',
    'razaosocialrfb', 'nomefantasiarfb', 'cnaeprincipal', 'desccnaeprincipal',
    'atendida', 'codigoassunto', 'descricaoassunto', 'codigoproblema',
    'descricaoproblema', 'sexoconsumidor', 'faixaetariaconsumidor', 'cepconsumidor'
]


# Criando um dicionário com todos os caminhos dos arquivos, baseando-se no ano de cada arquivo
arquivos = {
    '2009': '../dados_brutos/CNRF_2009.csv',
    '2010': '../dados_brutos/CNRF_2010.csv',
    '2011': '../dados_brutos/CNRF_2011.csv',
    '2012': '../dados_brutos/CNRF_2012.csv',
    '2013': '../dados_brutos/CNRF_2013.csv',
    '2014': '../dados_brutos/CNRF_2014.csv',
    '2015': '../dados_brutos/CNRF_2015.csv',
    '2016': '../dados_brutos/CNRF_2016.csv',
    '2017': '../dados_brutos/CNRF_2017.csv',
    '2018': '../dados_brutos/CNRF_2018.csv',
    '2019': '../dados_brutos/CNRF_2019.csv',
    '2020': '../dados_brutos/CNRF_2020.csv',
    '2021': '../dados_brutos/CNRF_2021.csv',
    '2022': '../dados_brutos/CNRF_2022.csv',
    '2023': '../dados_brutos/CNRF_2023.csv',
    '2024': '../dados_brutos/CNRF_2024.xlsx'
}

lista_dataframes = []                       # Para armazenar os dataframes até podermos agregá-los em um só
total_linhas_brutas = 0                     # Contadores para aferirmos se estamos perdendo muitas linhas
total_linhas_limpas = 0

print("Iniciando a Extração de Dados dos arquivos CNRF...\n")

for ano, caminho_arquivo in arquivos.items():
    linhas_no_arquivo   = 0
    linhas_lidas_pandas = 0
    
    try:
        ### CONTAGEM BRUTA
        if caminho_arquivo.endswith('.csv'):
            # Lendo o arquivo como texto puro apenas para contar as linhas
            with open(caminho_arquivo, 'r', encoding='latin1') as f:
                linhas_no_arquivo = sum(1 for linha in f)
            
            # Subtraindo a primeira linha (header), exceto para 2017 que não tem 
            if ano != '2017':
                linhas_no_arquivo -= 1
                
        elif caminho_arquivo.endswith('.xlsx'):
            # No casos especial do arquivo em excel, vamos deixar o Pandas lidar sozinho com a contagem bruta
            df_temp = pd.read_excel(caminho_arquivo, dtype=str)
            linhas_no_arquivo = len(df_temp)

        ### LEITURA E LIMPEZA INICIAL COM PANDAS
        if caminho_arquivo.endswith('.csv'):
            if ano == '2017':
                # Caso especial em que o arquivo não apresenta header
                df = pd.read_csv(caminho_arquivo, encoding='latin1', sep=';', header=None, names=colunas_padrao, dtype=str, on_bad_lines='skip')
            else:
                # Caso comum em que o arquivo apresenta header
                df = pd.read_csv(caminho_arquivo, encoding='latin1', sep=';', dtype=str, on_bad_lines='skip')
                df.columns = colunas_padrao # Forçando o padrão
                
        elif caminho_arquivo.endswith('.xlsx'):
            df = df_temp # Aproveitando que o arquivo já foi lido
            df.columns = colunas_padrao # Forçando o padrão
        
        ### FINALIZAÇÃO DO ARQUIVO
        df['ano_origem'] = ano
        lista_dataframes.append(df)
        
        linhas_lidas_pandas = len(df)
        linhas_perdidas = linhas_no_arquivo - linhas_lidas_pandas
        
        # Atualização dos contadores globais
        total_linhas_brutas += linhas_no_arquivo
        total_linhas_limpas += linhas_lidas_pandas
        
        print(f"[{ano}] - Brutas: {linhas_no_arquivo} | Salvas: {linhas_lidas_pandas} | Perdidas: {linhas_perdidas}")

    except Exception as e:
        print(f"[{ano}] - Erro Crítico: {e}")

### PRINT DOS RESULTADOS DA AUDITORIA
df_final = pd.concat(lista_dataframes, ignore_index=True)

taxa_perda = ((total_linhas_brutas - total_linhas_limpas) / total_linhas_brutas) * 100 if total_linhas_brutas > 0 else 0

print("-" * 40)
print("RESUMO DA EXTRAÇÃO")
print(f"Total de registros originais: {total_linhas_brutas}")
print(f"Total de registros recuperados: {total_linhas_limpas}")
print(f"Total de registros descartados: {total_linhas_brutas - total_linhas_limpas}")
print(f"Taxa de perda de dados: {taxa_perda:.4f}%")
print("-" * 40)

Iniciando a Extração de Dados dos arquivos CNRF...

[2009] - Brutas: 104869 | Salvas: 104869 | Perdidas: 0
[2010] - Brutas: 122664 | Salvas: 122664 | Perdidas: 0
[2011] - Brutas: 153096 | Salvas: 153096 | Perdidas: 0
[2012] - Brutas: 211076 | Salvas: 211076 | Perdidas: 0
[2013] - Brutas: 268096 | Salvas: 268096 | Perdidas: 0
[2014] - Brutas: 267764 | Salvas: 267764 | Perdidas: 0
[2015] - Brutas: 255650 | Salvas: 255650 | Perdidas: 0
[2016] - Brutas: 203485 | Salvas: 203485 | Perdidas: 0
[2017] - Brutas: 42307 | Salvas: 42307 | Perdidas: 0
[2018] - Brutas: 39060 | Salvas: 39047 | Perdidas: 13
[2019] - Brutas: 17577 | Salvas: 17555 | Perdidas: 22
[2020] - Brutas: 8016 | Salvas: 8006 | Perdidas: 10
[2021] - Brutas: 9007 | Salvas: 9007 | Perdidas: 0
[2022] - Brutas: 68303 | Salvas: 68289 | Perdidas: 14
[2023] - Brutas: 17266 | Salvas: 17266 | Perdidas: 0
[2024] - Brutas: 13803 | Salvas: 13803 | Perdidas: 0
----------------------------------------
RESUMO DA EXTRAÇÃO
Total de registros origi

## Transformação

In [8]:
# Verificando o resultado final da extração
with pd.option_context('display.max_columns', None):
    display(df_final.head(5))
    display(df_final.info())

,anocalendario,dataarquivamento,dataabertura,codigoregiao,regiao,uf,strrazaosocial,strnomefantasia,tipo,numerocnpj,radicalcnpj,razaosocialrfb,nomefantasiarfb,cnaeprincipal,desccnaeprincipal,atendida,codigoassunto,descricaoassunto,codigoproblema,descricaoproblema,sexoconsumidor,faixaetariaconsumidor,cepconsumidor,ano_origem
0,2009,2009-01-21 15:29:29.000,2005-07-06 08:32:23.000,05,Centro-oeste,GO,CBP SUL - COLCHÕES E ESPUMAS INDUSTRIAIS LTDA,LIMANSKY,1,01350934000116,01350934,CBP SUL - COLCHOES E ESPUMAS INDUSTRIAIS LTDA,NaN,3104700,FABRICAÇÃO DE COLCHÕES,S,100,Colchão,105,Produto entregue com danos/defeitos,M,entre 61 a 70 anos,74680330,2009
1,2009,2009-05-12 12:05:08.000,2006-01-17 12:10:53.000,03,Sudeste,RJ,GRADIENTE,NaN,1,43185362001936,43185362,IGB ELETRONICA S.A,NaN,2640000,"FABRICAÇÃO DE APARELHOS DE RECEPÇÃO, REPRODUÇÃ...",N,146,Aparelho DVD,107,Não entrega/demora na entrega do produto,F,entre 51 a 60 anos,23510240,2009
2,2009,2009-04-23 08:52:31.000,2005-07-06 11:28:47.000,05,Centro-oeste,GO,IGL INDÚSTRIA LTDA,ELIDA PONDS INDUSTRIAL LTDA,1,03085759000102,03085759,UNILEVER BRASIL HIGIENE PESSOAL E LIMPEZA LTDA,ELIDA PONDS INDUSTRIAL LTDA.,2063100,"FABRICAÇÃO DE COSMÉTICOS, PRODUTOS DE PERFUMAR...",N,224,Produto Para Uso Veterinário ( Medicamento / S...,177,Presença de sujidades/corpos estranhos,F,entre 21 a 30 anos,74775010,2009
3,2009,2009-04-23 08:52:31.000,2005-07-06 11:28:47.000,05,Centro-oeste,GO,UNILEVER BESTFOODS BRASIL LTDA,UNILEVER,1,01615814002066,01615814,UNILEVER BRASIL INDUSTRIAL LTDA,NaN,1031700,FABRICAÇÃO DE CONSERVAS DE FRUTAS,N,224,Produto Para Uso Veterinário ( Medicamento / S...,177,Presença de sujidades/corpos estranhos,F,entre 21 a 30 anos,74775010,2009
4,2009,2009-07-23 11:57:40.000,2006-01-18 15:35:29.000,03,Sudeste,RJ,EMPRESA BRASILEIRA DE TELECOMUNICAÇÕES S.A,EMBRATEL - LIVRE,1,33530486000129,33530486,EMPRESA BRASILEIRA DE TELECOMUNICACOES S A EMB...,NaN,6110899,SERVIÇOS DE TELECOMUNICAÇÕES POR FIO NÃO ESPEC...,S,186,Telefonia Fixa ( Plano de Expansão / Compra e ...,143,Contrato - Rescisão/alteração unilateral,F,entre 41 a 50 anos,20710270,2009


<class 'pandas.DataFrame'>
RangeIndex: 1801980 entries, 0 to 1801979
Data columns (total 24 columns):
 #   Column                 Dtype
---  ------                 -----
 0   anocalendario          str  
 1   dataarquivamento       str  
 2   dataabertura           str  
 3   codigoregiao           str  
 4   regiao                 str  
 5   uf                     str  
 6   strrazaosocial         str  
 7   strnomefantasia        str  
 8   tipo                   str  
 9   numerocnpj             str  
 10  radicalcnpj            str  
 11  razaosocialrfb         str  
 12  nomefantasiarfb        str  
 13  cnaeprincipal          str  
 14  desccnaeprincipal      str  
 15  atendida               str  
 16  codigoassunto          str  
 17  descricaoassunto       str  
 18  codigoproblema         str  
 19  descricaoproblema      str  
 20  sexoconsumidor         str  
 21  faixaetariaconsumidor  str  
 22  cepconsumidor          str  
 23  ano_origem             str  
dtypes: str(24

None